# 🏆 DATATHON 2026 — GenCore v7 | NaiveKing

**Strategy từ LB evidence:**
- v5 naive=931k, v6 XGB+ProphetAux=1.1M → Prophet aux HURTS
- v7: Naive364 backbone + XGB/LGB chỉ correct residuals từ CLEAN features
- NO Prophet anywhere. Aux features = historical median profiles only.
- Proper 548-day holdout CV để tune correction strength

**Upload to Kaggle → Run All → submit submission_v7_best.csv**

In [ ]:
# ─── SETUP ───
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

KAGGLE = os.path.exists('/kaggle/input')
if KAGGLE:
    matches = glob.glob('/kaggle/input/**/sales.csv', recursive=True)
    DATA_DIR = os.path.dirname(matches[0]) if matches else '/kaggle/input'
    OUT_DIR = '/kaggle/working'
else:
    DATA_DIR = 'data/raw'
    for c in ['data/raw', '../data/raw']:
        if os.path.isfile(os.path.join(c, 'sales.csv')): DATA_DIR = c; break
    OUT_DIR = 'output'
os.makedirs(OUT_DIR, exist_ok=True)

try:
    from lightgbm import LGBMRegressor
except ImportError:
    os.system('pip install -q lightgbm'); from lightgbm import LGBMRegressor
try:
    from xgboost import XGBRegressor
except ImportError:
    os.system('pip install -q xgboost'); from xgboost import XGBRegressor

print(f"ENV: {'Kaggle' if KAGGLE else 'Local'} | DATA: {DATA_DIR}")

sales = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'))
sub_tpl = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
sales['Date'] = pd.to_datetime(sales['Date'])
sub_tpl['Date'] = pd.to_datetime(sub_tpl['Date'])
sales = sales.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
sub_tpl = sub_tpl.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
forecast_dates = sub_tpl['Date'].tolist()
N_FC = len(forecast_dates)
print(f"Train: {sales.Date.min().date()} -> {sales.Date.max().date()} ({len(sales)} rows)")
print(f"Forecast: {N_FC} days ({forecast_dates[0].date()} -> {forecast_dates[-1].date()})")

In [ ]:
# ─── CALENDAR & HOLIDAYS ───
TET = pd.to_datetime([
    '2012-01-23','2013-02-10','2014-01-31','2015-02-19','2016-02-08',
    '2017-01-28','2018-02-16','2019-02-05','2020-01-25','2021-02-12',
    '2022-02-01','2023-01-22','2024-02-10'
])
MEGA_SALES = [(1,1),(3,3),(4,30),(5,1),(6,6),(7,7),(8,8),(9,2),(9,9),(10,10),(11,11),(12,12)]

def get_tet_for_year(year):
    for t in TET:
        if t.year == year: return t
    return None

def days_to_next_tet(dates):
    tet_arr = np.sort(np.array(TET, dtype='datetime64[ns]'))
    d_arr = np.array(pd.to_datetime(dates), dtype='datetime64[ns]')
    out = np.full(len(d_arr), 365, dtype=int)
    idx = np.searchsorted(tet_arr, d_arr, side='left')
    for i in range(len(d_arr)):
        if idx[i] < len(tet_arr):
            out[i] = int((tet_arr[idx[i]] - d_arr[i]) / np.timedelta64(1,'D'))
    return np.clip(out, 0, 365)

def days_since_last_tet(dates):
    tet_arr = np.sort(np.array(TET, dtype='datetime64[ns]'))
    d_arr = np.array(pd.to_datetime(dates), dtype='datetime64[ns]')
    out = np.full(len(d_arr), 365, dtype=int)
    idx = np.searchsorted(tet_arr, d_arr, side='right') - 1
    for i in range(len(d_arr)):
        if idx[i] >= 0:
            out[i] = int((d_arr[i] - tet_arr[idx[i]]) / np.timedelta64(1,'D'))
    return np.clip(out, 0, 365)

def is_mega_sale(dates, window=2):
    dates = pd.to_datetime(dates)
    out = np.zeros(len(dates), dtype=int)
    for i, d in enumerate(dates):
        for m, day in MEGA_SALES:
            try:
                sale_d = pd.Timestamp(year=d.year, month=m, day=day)
                if abs((d - sale_d).days) <= window:
                    out[i] = 1; break
            except: pass
    return out

print('Calendar helpers ready')

In [ ]:
# ─── AUX FEATURES (clean — NO Prophet) ───
aux_profiles = {}

for fn, dc, cols in [
    ('web_traffic.csv', 'date', ['sessions', 'unique_visitors', 'page_views']),
    ('orders.csv', 'order_date', [])
]:
    fp = os.path.join(DATA_DIR, fn)
    if not os.path.isfile(fp): continue
    df = pd.read_csv(fp)
    df[dc] = pd.to_datetime(df[dc], errors='coerce')
    df = df.dropna(subset=[dc])
    df['month'] = df[dc].dt.month
    df['dow']   = df[dc].dt.dayofweek
    if cols:
        for c in cols:
            if c in df.columns:
                aux_profiles[f'{c}_month'] = df.groupby('month')[c].median().to_dict()
                aux_profiles[f'{c}_dow']   = df.groupby('dow')[c].median().to_dict()
    else:
        daily = df.groupby(dc).size().reset_index(name='n')
        daily['month'] = daily[dc].dt.month
        daily['dow']   = daily[dc].dt.dayofweek
        aux_profiles['orders_month'] = daily.groupby('month')['n'].median().to_dict()
        aux_profiles['orders_dow']   = daily.groupby('dow')['n'].median().to_dict()

print(f'Aux profiles: {list(aux_profiles.keys())}')

In [ ]:
# ─── NAIVE MODELS (BACKBONE) ───

def naive364(train_series, fc_dates):
    s = train_series.sort_index().astype(float)
    h = {pd.Timestamp(i): float(v) for i, v in s.items()}
    fallback = float(s.tail(28).median())
    preds = []
    for dt in pd.to_datetime(fc_dates):
        val = None
        for off in [364, 371, 357, 728, 735, 721]:
            c = dt - pd.Timedelta(days=off)
            if c in h: val = h[c]; break
        if val is None: val = fallback
        h[dt] = float(val)
        preds.append(float(val))
    return np.array(preds)


def naive_median_blend(train_df, col, fc_dates, n_years=4):
    work = train_df[['Date', col]].copy()
    work['doy'] = work.Date.dt.dayofyear
    work['dow'] = work.Date.dt.dayofweek
    recent_years = sorted(work.Date.dt.year.unique())[-n_years:]
    work = work[work.Date.dt.year.isin(recent_years)]
    preds = []
    for dt in pd.to_datetime(fc_dates):
        doy = dt.dayofyear
        dow = dt.dayofweek
        cands = work[(abs(work.doy - doy) <= 3) & (work.dow == dow)][col]
        if len(cands) >= 2:
            preds.append(float(np.median(cands.values)))
        else:
            fb = work[work.dow == dow][col].median()
            preds.append(float(fb) if not np.isnan(fb) else float(work[col].median()))
    return np.array(preds)


def growth_adjusted_naive(train_df, col, fc_dates):
    yearly = train_df.groupby(train_df.Date.dt.year)[col].median()
    if len(yearly) >= 3:
        recent = yearly.iloc[-3:].values
        ratios = recent[1:] / np.maximum(recent[:-1], 1)
        growth = float(np.median(ratios))
        growth = np.clip(growth, 0.9, 1.3)
    else:
        growth = 1.0
    base = naive364(train_df.set_index('Date')[col], fc_dates)
    max_train_year = train_df.Date.dt.year.max()
    fc_years = np.array([pd.Timestamp(d).year for d in pd.to_datetime(fc_dates)], dtype=float)
    year_diff = fc_years - max_train_year
    multiplier = growth ** np.clip(year_diff, 0, 3)
    return base * multiplier


def blend_naive(p364, p_med, p_grow, w=(0.5, 0.3, 0.2)):
    return w[0]*p364 + w[1]*p_med + w[2]*p_grow


print('Naive models ready')

In [ ]:
# ─── TET EMPIRICAL CALIBRATION ───

def compute_tet_multipliers(sales_df, col):
    mults = {}
    for tet in TET:
        yr_data = sales_df[sales_df.Date.dt.year == tet.year]
        if len(yr_data) < 100: continue
        baseline = yr_data[yr_data.Date.dt.month.isin([5,6,7,8])][col].median()
        if baseline <= 0: continue
        for delta in range(-30, 21):
            d = tet + pd.Timedelta(days=delta)
            row = sales_df[sales_df.Date == d]
            if len(row) > 0:
                mults.setdefault(delta, []).append(float(row[col].iloc[0]) / baseline)
    return {k: float(np.median(v)) for k, v in mults.items() if len(v) >= 2}


def apply_tet_calibration(preds, fc_dates, tet_mults, base_median, strength=0.4):
    preds = preds.copy()
    fc_dates_ts = pd.to_datetime(fc_dates)
    for i, dt in enumerate(fc_dates_ts):
        tet = get_tet_for_year(dt.year)
        if tet is None: continue
        delta = (dt - tet).days
        if delta in tet_mults:
            target = base_median * tet_mults[delta]
            preds[i] = (1 - strength) * preds[i] + strength * target
    return np.clip(preds, 0, None)


print('Tet calibration ready')

In [ ]:
# ─── FEATURE ENGINEERING (NO Prophet, NO recursive lags) ───

def build_features_v7(fc_dates, train_df, col, aux_profiles):
    dates = pd.to_datetime(fc_dates)
    n = len(dates)
    feats = pd.DataFrame(index=range(n))

    feats['month']          = dates.month
    feats['day']            = dates.day
    feats['dow']            = dates.dayofweek
    feats['doy']            = dates.dayofyear
    feats['week']           = dates.isocalendar().week.astype(int).values
    feats['quarter']        = dates.quarter
    feats['year_rel']       = dates.year - 2012
    feats['is_weekend']     = (dates.dayofweek >= 5).astype(int)
    feats['is_month_start'] = dates.is_month_start.astype(int)
    feats['is_month_end']   = dates.is_month_end.astype(int)
    feats['is_payday']      = dates.day.isin([1, 15, 30]).astype(int)
    feats['is_tet_month']   = dates.month.isin([1, 2]).astype(int)
    feats['is_mega_sale']   = is_mega_sale(dates)

    feats['days_to_tet']    = days_to_next_tet(dates)
    feats['days_since_tet'] = days_since_last_tet(dates)
    feats['tet_window']     = (feats['days_to_tet'] <= 10).astype(int)
    feats['post_tet']       = ((feats['days_since_tet'] > 0) & (feats['days_since_tet'] <= 14)).astype(int)
    feats['tet_effect']     = np.exp(-feats['days_to_tet'] / 7) + np.exp(-feats['days_since_tet'] / 5)

    for k in [1, 2, 3, 4]:
        feats[f'sin_y{k}'] = np.sin(2*np.pi*k * feats['doy'] / 365.25)
        feats[f'cos_y{k}'] = np.cos(2*np.pi*k * feats['doy'] / 365.25)
    for k in [1, 2]:
        feats[f'sin_w{k}'] = np.sin(2*np.pi*k * feats['dow'] / 7)
        feats[f'cos_w{k}'] = np.cos(2*np.pi*k * feats['dow'] / 7)

    # Fixed-lookback lag features (into training data only)
    idx_map = train_df.set_index('Date')[col]
    for lag_year in [1, 2, 3]:
        lag_vals = []
        for dt in dates:
            hits = []
            for off in [364*lag_year, 364*lag_year+7, 364*lag_year-7]:
                c = dt - pd.Timedelta(days=off)
                if c in idx_map.index:
                    hits.append(float(idx_map[c]))
            lag_vals.append(float(np.mean(hits)) if hits else np.nan)
        feats[f'lag_{lag_year}yr'] = lag_vals
    
    # Rolling stats from training (fixed, no recursion)
    for w_days in [28, 90, 182]:
        cutoff = dates.min() - pd.Timedelta(days=1)
        tail = train_df[train_df.Date <= cutoff].tail(w_days)
        feats[f'train_tail{w_days}_mean'] = tail[col].mean() if len(tail) else 0
        feats[f'train_tail{w_days}_med']  = tail[col].median() if len(tail) else 0

    # Historical month/DOW stats
    monthly = train_df.groupby(train_df.Date.dt.month)[col].agg(['mean','median','std'])
    feats['hist_month_mean'] = feats['month'].map(monthly['mean'])
    feats['hist_month_med']  = feats['month'].map(monthly['median'])
    feats['hist_month_std']  = feats['month'].map(monthly['std'])

    dow_stats = train_df.groupby(train_df.Date.dt.dayofweek)[col].agg(['mean','median'])
    feats['hist_dow_mean'] = feats['dow'].map(dow_stats['mean'])
    feats['hist_dow_med']  = feats['dow'].map(dow_stats['median'])

    # Interaction: month x DOW
    md = train_df.groupby([train_df.Date.dt.month, train_df.Date.dt.dayofweek])[col].median()
    feats['hist_month_dow'] = [
        md.get((feats['month'].iloc[i], feats['dow'].iloc[i]), np.nan)
        for i in range(n)
    ]

    # Aux profiles
    for k, v in aux_profiles.items():
        if '_month' in k:
            feats[f'aux_{k}'] = feats['month'].map(v)
        elif '_dow' in k:
            feats[f'aux_{k}'] = feats['dow'].map(v)

    return feats.fillna(0)


print('Feature engineering ready')

In [ ]:
# ─── V7 PIPELINE ───

def run_v7_pipeline(train_df, col, fc_dates, aux_profiles, tet_strength=0.35):
    fc_dates = list(pd.to_datetime(fc_dates))
    n = len(fc_dates)

    # Step 1: Naive backbone
    p364   = naive364(train_df.set_index('Date')[col], fc_dates)
    p_med  = naive_median_blend(train_df, col, fc_dates, n_years=4)
    p_grow = growth_adjusted_naive(train_df, col, fc_dates)
    p_naive = blend_naive(p364, p_med, p_grow, w=(0.5, 0.3, 0.2))

    # Step 2: Residual correction
    # Fit on last 2.5 years of training
    cutoff = train_df.Date.max() - pd.DateOffset(years=2, months=6)
    hist = train_df[train_df.Date >= cutoff].copy()

    idx_map = train_df.set_index('Date')[col]
    naive_hist = []
    for dt in hist.Date:
        val = None
        for off in [364, 371, 357, 728]:
            c = dt - pd.Timedelta(days=off)
            if c in idx_map.index:
                val = float(idx_map[c]); break
        naive_hist.append(val if val is not None else float(idx_map.median()))

    hist = hist.copy()
    hist['naive_pred'] = naive_hist
    hist['residual'] = hist[col] - hist['naive_pred']

    X_tr = build_features_v7(hist.Date.tolist(), train_df, col, aux_profiles)
    y_resid = hist['residual'].values

    # Clip extreme residuals
    clip = np.percentile(np.abs(y_resid), 97)
    mask = np.abs(y_resid) <= clip * 2
    X_tr_c, y_tr_c = X_tr[mask], y_resid[mask]

    lgb = LGBMRegressor(
        n_estimators=800, learning_rate=0.02, num_leaves=31, max_depth=6,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=10.0,
        min_child_samples=30, random_state=SEED, n_jobs=-1, verbosity=-1
    )
    xgb = XGBRegressor(
        n_estimators=800, learning_rate=0.02, max_depth=5,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=10.0,
        min_child_weight=5, random_state=SEED, n_jobs=-1, verbosity=0
    )
    lgb.fit(X_tr_c, y_tr_c)
    xgb.fit(X_tr_c, y_tr_c)

    X_fc = build_features_v7(fc_dates, train_df, col, aux_profiles)
    corr = 0.6 * lgb.predict(X_fc) + 0.4 * xgb.predict(X_fc)

    # Damp at far horizons
    damp = np.exp(-np.arange(n, dtype=float) / 300)
    p_corrected = np.clip(p_naive + corr * damp, 0, None)

    # Step 3: Tet calibration
    tet_mults = compute_tet_multipliers(train_df, col)
    base_med = float(train_df[train_df.Date.dt.month.isin([5,6,7,8])][col].median())

    p_naive_tet     = apply_tet_calibration(p_naive,     fc_dates, tet_mults, base_med, tet_strength)
    p_corrected_tet = apply_tet_calibration(p_corrected, fc_dates, tet_mults, base_med, tet_strength)

    return {
        'naive':         np.clip(p_naive, 0, None),
        'corrected':     p_corrected,
        'naive_tet':     p_naive_tet,
        'corrected_tet': p_corrected_tet,
        'correction':    corr,
        'damp':          damp,
    }


print('V7 pipeline ready')

In [ ]:
# ─── 548-DAY HOLDOUT CROSS-VALIDATION ───
print('\n' + '='*70)
print('548-DAY HOLDOUT VALIDATION')
print('='*70)

HOLDOUTS = [
    ('2021-06-30', 548),
    ('2021-01-01', 548),
    ('2020-06-30', 548),
]

cv_all = {}

for col in ['Revenue', 'COGS']:
    print(f'\n-- {col} --')
    fold_results = {k: [] for k in ['naive','corrected','naive_tet','corrected_tet']}

    for origin_str, n_days in HOLDOUTS:
        origin = pd.Timestamp(origin_str)
        tr = sales[sales.Date <= origin].copy()
        val = sales[sales.Date > origin].head(n_days).copy()
        if len(val) < 100: continue

        y_true = val[col].values
        preds = run_v7_pipeline(tr, col, val.Date.tolist(), aux_profiles)

        for k in fold_results:
            mae = mean_absolute_error(y_true, preds[k])
            fold_results[k].append(mae)
            print(f'  {origin_str} | {k}: MAE={mae:,.0f}')

    print(f'  {col} WEIGHTED AVG:')
    for k, maes in fold_results.items():
        if maes:
            w = np.array([1.0 + i*0.5 for i in range(len(maes))]); w /= w.sum()
            avg = float(np.dot(maes, w))
            cv_all[f'{col}_{k}'] = avg
            print(f'    {k}: {avg:,.0f}')

In [ ]:
# ─── TUNE TET STRENGTH ───
print('\n' + '='*70)
print('TUNING TET STRENGTH')
print('='*70)

origin = pd.Timestamp('2021-06-30')
tr_t = sales[sales.Date <= origin].copy()
val_t = sales[sales.Date > origin].head(548).copy()

best_tet_str = {}
for col in ['Revenue', 'COGS']:
    y = val_t[col].values
    preds_base = run_v7_pipeline(tr_t, col, val_t.Date.tolist(), aux_profiles, tet_strength=0.0)
    tet_mults = compute_tet_multipliers(tr_t, col)
    base_med = float(tr_t[tr_t.Date.dt.month.isin([5,6,7,8])][col].median())

    best_mae, best_s = 1e18, 0.35
    print(f'\n{col}:')
    for s in [0.0, 0.15, 0.25, 0.35, 0.45, 0.55]:
        p = apply_tet_calibration(preds_base['corrected'], val_t.Date.tolist(), tet_mults, base_med, s)
        mae = mean_absolute_error(y, p)
        print(f'  tet={s}: MAE={mae:,.0f}')
        if mae < best_mae: best_mae, best_s = mae, s
    best_tet_str[col] = best_s
    print(f'  -> Best: {best_s}')

print('\nTuning done')

In [ ]:
# ─── FINAL FORECAST ───
print('\n' + '='*70)
print('FINAL FORECAST')
print('='*70)

final_preds = {}
for col in ['Revenue', 'COGS']:
    print(f'  {col}...', end=' ', flush=True)
    t0 = time.time()
    final_preds[col] = run_v7_pipeline(
        sales, col, forecast_dates, aux_profiles,
        tet_strength=best_tet_str.get(col, 0.35)
    )
    print(f'done ({time.time()-t0:.0f}s)')

In [ ]:
# ─── BUILD & SAVE SUBMISSIONS ───
print('\n' + '='*70)
print('SUBMISSIONS')
print('='*70)

VARIANTS = ['naive', 'corrected', 'naive_tet', 'corrected_tet']
output_files = {}

for vname in VARIANTS:
    sub = sub_tpl[['Date']].copy()
    for col in ['Revenue', 'COGS']:
        sub[col] = np.clip(final_preds[col][vname], 0, None)

    ok = (len(sub) == N_FC
          and not sub[['Revenue','COGS']].isna().any().any()
          and not (sub[['Revenue','COGS']] < 0).any().any())
    print(f"  {'OK' if ok else 'FAIL'} {vname}: {len(sub)} rows")
    for c in ['Revenue','COGS']:
        v = sub[c]
        print(f'    {c}: mean={v.mean():,.0f} med={v.median():,.0f} min={v.min():,.0f} max={v.max():,.0f}')

    fname = f'submission_v7_{vname}.csv'
    fpath = os.path.join(OUT_DIR, fname)
    out = sub.copy()
    out['Date'] = pd.to_datetime(out['Date']).dt.strftime('%Y-%m-%d')
    out.to_csv(fpath, index=False)
    output_files[vname] = fpath

# Diagnostics
diag = {
    'version': 'v7_naiveking',
    'cv_results': cv_all,
    'best_tet_strength': best_tet_str,
    'strategy': 'Naive364 backbone + XGB/LGB residual (no Prophet anywhere)',
}
with open(os.path.join(OUT_DIR, 'v7_diagnostics.json'), 'w') as f:
    json.dump(diag, f, indent=2, default=str)

print('\n' + '='*70)
print('SUBMIT ORDER:')
print('  1. submission_v7_corrected_tet.csv  <- try first')
print('  2. submission_v7_corrected.csv')
print('  3. submission_v7_naive_tet.csv')
print('  4. submission_v7_naive.csv          <- pure baseline')
print(f'\nCurrent best on LB: 931,624 (v5). Target: <800k')